# 03 — Network Analysis

Khảo sát cấu trúc mạng, độ quan trọng của sản phẩm và các cộng đồng để nhận diện quan hệ giữa sản phẩm.


## 1. Objective & Graph Loading

Trả lời RQ1 về các nhóm sản phẩm thường mua chung và RQ2 về vai trò sản phẩm trong mạng. Đọc graph chính thức và các bảng Phase 2; không xử lý lại giao dịch. Baseline luôn là graph đã lưu theo config, các ứng viên chỉ tồn tại trong bộ nhớ.

In [1]:
from pathlib import Path
from collections import Counter
from itertools import combinations
from math import comb
import hashlib
import json
import sys

import numpy as np
import pandas as pd
import networkx as nx
from IPython.display import display

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))
from config import TABLES_DIR, NETWORKS_DIR, TOP_N_PRODUCTS, MIN_EDGE_WEIGHT, RANDOM_STATE

G = nx.read_graphml(NETWORKS_DIR / "product_network.graphml")
phase2 = json.loads((NETWORKS_DIR / "network_summary.json").read_text())
products = pd.read_csv(TABLES_DIR / "selected_products.csv", dtype={"StockCode": "string"})
products = products.sort_values(["invoice_count", "StockCode"], ascending=[False, True])
pairs = pd.read_csv(TABLES_DIR / "pair_counts.csv", dtype={"ProductA": "string", "ProductB": "string"})
assert phase2["configuration"]["TOP_N_PRODUCTS"] == TOP_N_PRODUCTS and phase2["configuration"]["MIN_EDGE_WEIGHT"] == MIN_EDGE_WEIGHT, "Config differs from saved graph: rerun Phase 2 first."
assert not G.is_directed() and not G.is_multigraph(), "An undirected simple graph is required."
weights = np.array([d.get("weight", np.nan) for _, _, d in G.edges(data=True)], dtype=float)
assert len(weights) and np.isfinite(weights).all() and (weights > 0).all(), "Louvain requires positive finite weights and at least one edge."

resolution = 1.0
seeds = [RANDOM_STATE + i for i in range(10)]
baseline_name = f"baseline_top{TOP_N_PRODUCTS}_t{MIN_EDGE_WEIGHT}"
baseline_info = {"config_name": baseline_name, "top_n": TOP_N_PRODUCTS, "threshold": MIN_EDGE_WEIGHT, "random_state": RANDOM_STATE, "resolution": resolution}
display(pd.Series({**baseline_info, "nodes": len(G), "edges": G.number_of_edges(), "density": nx.density(G), "networkx_version": nx.__version__}, name="Official baseline"))

config_name         baseline_top100_t125
top_n                                100
threshold                            125
random_state                          42
resolution                           1.0
nodes                                100
edges                               2074
density                          0.41899
networkx_version                     3.7
Name: Official baseline, dtype: object

## 2. Louvain Community Detection

Community detection tìm các nhóm có liên kết nội bộ mạnh hơn kỳ vọng từ phân bố strength. Louvain thử chuyển node để tăng weighted modularity, gộp cộng đồng thành node rồi lặp lại. Giữ resolution = 1 và dùng RANDOM_STATE; không dò tham số để tối đa hóa một chỉ số.

Modularity đo mức liên kết nội bộ vượt mô hình kỳ vọng bảo toàn strength. Đây là tiêu chí của thuật toán, không chứng minh nhóm có ý nghĩa thương mại; kết quả có thể phụ thuộc seed và bỏ sót nhóm nhỏ. Xem [phương pháp Louvain trong NetworkX](https://networkx.org/documentation/stable/reference/algorithms/generated/networkx.algorithms.community.louvain.louvain_communities.html).

In [2]:
def louvain_partition(graph, seed):
    groups = nx.community.louvain_communities(graph, weight="weight", resolution=resolution, seed=seed)
    groups = sorted(groups, key=lambda group: min(group))
    assert sum(map(len, groups)) == len(graph) and set().union(*groups) == set(graph), "Each node must belong to exactly one community."
    return groups


def membership(groups):
    return {node: i + 1 for i, group in enumerate(groups) for node in group}


communities = louvain_partition(G, RANDOM_STATE)
community_ids = membership(communities)
modularity = nx.community.modularity(G, communities, weight="weight", resolution=resolution)
assert np.isfinite(modularity) and -1 <= modularity <= 1
display(pd.Series({"communities": len(communities), "weighted_modularity": modularity, "community_sizes": [len(group) for group in communities]}, name=baseline_name))

communities                              5
weighted_modularity               0.184622
community_sizes        [18, 21, 44, 4, 13]
Name: baseline_top100_t125, dtype: object

## 3. Community Interpretation & Stability

Mỗi cộng đồng được mô tả bằng StockCode và Description, không dùng Description làm định danh. Sản phẩm nổi bật trong bảng dưới được chọn theo số hóa đơn từ Phase 2. Mỗi cạnh nội bộ được cộng một lần; cạnh nối hai cộng đồng đóng góp vào external weight của cả hai cộng đồng.

ARI so sánh phân hoạch độc lập với tên nhãn: 1 là trùng nhau, khoảng 0 là mức kỳ vọng ngẫu nhiên, giá trị âm là kém hơn kỳ vọng. Tính ARI bằng số cặp trong bảng chéo của hai phân hoạch, theo [định nghĩa Adjusted Rand Index](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.adjusted_rand_score.html). Chạy 10 seeds liên tiếp bắt đầu từ RANDOM_STATE; mean/min pairwise ARI dùng 45 cặp runs khác nhau. Đây là ổn định theo seed trên cùng graph, chưa phải độ tin cậy trên dữ liệu mới.

In [3]:
community_rows = []
for community_id, group in enumerate(communities, start=1):
    members = products.loc[products["StockCode"].isin(group)]
    community_rows.append({
        **baseline_info,
        "community_id": community_id,
        "size": len(group),
        "internal_edge_weight": int(G.subgraph(group).size(weight="weight")),
        "external_edge_weight": sum(d["weight"] for u, v, d in G.edges(data=True) if (u in group) != (v in group)),
        "products": json.dumps(members[["StockCode", "Description"]].to_dict("records"), ensure_ascii=False),
        "prominent_products_by_invoice_count": json.dumps(members.head(3)[["StockCode", "Description", "invoice_count"]].to_dict("records"), ensure_ascii=False),
    })
community_summary = pd.DataFrame(community_rows)
display(community_summary[["community_id", "size", "internal_edge_weight", "external_edge_weight", "prominent_products_by_invoice_count"]])

,community_id,size,internal_edge_weight,external_edge_weight,prominent_products_by_invoice_count
0,1,18,40042,112046,"[{""StockCode"": ""85099B"", ""Description"": ""JUMBO..."
1,2,21,49311,120997,"[{""StockCode"": ""20725"", ""Description"": ""LUNCH ..."
2,3,44,100940,136709,"[{""StockCode"": ""85123A"", ""Description"": ""WHITE..."
3,4,4,1768,4486,"[{""StockCode"": ""21181"", ""Description"": ""PLEASE..."
4,5,13,7726,37280,"[{""StockCode"": ""22086"", ""Description"": ""PAPER ..."


In [4]:
def adjusted_rand(labels_a, labels_b):
    assert len(labels_a) == len(labels_b)
    total_pairs = comb(len(labels_a), 2)
    if total_pairs == 0:
        return 1.0
    joint_pairs = sum(comb(n, 2) for n in Counter(zip(labels_a, labels_b)).values())
    a_pairs = sum(comb(n, 2) for n in Counter(labels_a).values())
    b_pairs = sum(comb(n, 2) for n in Counter(labels_b).values())
    expected = a_pairs * b_pairs / total_pairs
    upper = (a_pairs + b_pairs) / 2
    return (joint_pairs - expected) / (upper - expected) if upper != expected else 1.0


def stability_analysis(graph, reference):
    nodes = sorted(graph)
    reference_ids = membership(reference)
    reference_labels = [reference_ids[node] for node in nodes]
    labels_by_seed, rows = [], []
    for seed in seeds:
        groups = reference if seed == RANDOM_STATE else louvain_partition(graph, seed)
        ids = membership(groups)
        labels = [ids[node] for node in nodes]
        labels_by_seed.append(labels)
        rows.append({"seed": seed, "number_of_communities": len(groups),
                     "modularity": nx.community.modularity(graph, groups, weight="weight", resolution=resolution),
                     "ari_vs_reference": adjusted_rand(reference_labels, labels)})
    pairwise = [adjusted_rand(a, b) for a, b in combinations(labels_by_seed, 2)]
    runs = pd.DataFrame(rows)
    assert np.isfinite(runs[["modularity", "ari_vs_reference"]]).all().all() and runs["modularity"].between(-1, 1).all() and runs["ari_vs_reference"].between(-1, 1).all()
    repeat_ids = membership(louvain_partition(graph, RANDOM_STATE))
    assert adjusted_rand(reference_labels, [repeat_ids[node] for node in nodes]) == 1.0, "Same-seed partition must be reproducible."
    stats = {"mean_pairwise_ari": float(np.mean(pairwise)), "min_pairwise_ari": min(pairwise),
             "min_modularity": runs["modularity"].min(), "max_modularity": runs["modularity"].max(),
             "min_communities": runs["number_of_communities"].min(), "max_communities": runs["number_of_communities"].max()}
    return runs.assign(mean_pairwise_ari=stats["mean_pairwise_ari"], min_pairwise_ari=stats["min_pairwise_ari"]), stats


baseline_runs, baseline_stability = stability_analysis(G, communities)
display(baseline_runs)
display(pd.Series(baseline_stability, name="Baseline stability"))

,seed,number_of_communities,modularity,ari_vs_reference,mean_pairwise_ari,min_pairwise_ari
0,42,5,0.184622,1.000000,0.940358,0.796008
1,43,5,0.186281,0.967877,0.940358,0.796008
2,44,5,0.184622,1.000000,0.940358,0.796008
3,45,5,0.186281,0.967877,0.940358,0.796008
4,46,5,0.186281,0.967877,0.940358,0.796008
5,47,5,0.185732,0.796008,0.940358,0.796008
6,48,5,0.184622,1.000000,0.940358,0.796008
7,49,5,0.186281,0.967877,0.940358,0.796008
8,50,5,0.187033,0.907576,0.940358,0.796008
9,51,5,0.186281,0.967877,0.940358,0.796008


mean_pairwise_ari    0.940358
min_pairwise_ari     0.796008
min_modularity       0.184622
max_modularity       0.187033
min_communities      5.000000
max_communities      5.000000
Name: Baseline stability, dtype: float64

## 4. Centrality Analysis

Degree đếm số hàng xóm; strength cộng tần suất mua chung của các cạnh kề. Weighted betweenness đo vai trò trung gian trên các đường đi ngắn nhất với distance = 1 / weight, normalized và không tính endpoints. Khoảng cách nghịch đảo là giả định mô hình về độ gần, không phải luồng mua hàng đã quan sát.

Trong graph đầy đủ, mọi node có unweighted degree = n − 1 và unweighted betweenness = 0 vì mọi cặp có cạnh trực tiếp. Weighted betweenness vẫn có thể khác 0: đi qua hai liên kết mạnh có thể ngắn hơn một liên kết trực tiếp yếu. Tính chính xác trên toàn bộ nodes, không lấy mẫu; so sánh trong cùng phiên bản NetworkX. Khoảng cách số thực có thể ảnh hưởng nhận diện các đường đi bằng nhau.

In [5]:
def centrality_table(graph, groups):
    for _, _, data in graph.edges(data=True):
        data["distance"] = 1 / data["weight"]
    table = pd.DataFrame({
        "community_id": membership(groups),
        "degree": dict(graph.degree()),
        "strength": dict(graph.degree(weight="weight")),
        "betweenness": nx.betweenness_centrality(graph, weight="distance", normalized=True, endpoints=False),
    }).rename_axis("StockCode").reset_index()
    values = table[["degree", "strength", "betweenness"]]
    assert np.isfinite(values).all().all() and values.ge(0).all().all() and table["betweenness"].le(1 + 1e-12).all()
    table = table.merge(products, on="StockCode", how="left", validate="one_to_one")
    table["strength_rank"] = table["strength"].rank(ascending=False, method="average")
    table["betweenness_rank"] = table["betweenness"].rank(ascending=False, method="average")
    table["rank_gap"] = table["strength_rank"] - table["betweenness_rank"]
    return table.sort_values("StockCode").reset_index(drop=True)


node_metrics = centrality_table(G, communities).assign(**baseline_info)
metric_columns = ["StockCode", "Description", "invoice_count", "community_id", "degree", "strength", "betweenness"]
top_strength = node_metrics.sort_values(["strength", "StockCode"], ascending=[False, True]).head(10)
top_betweenness = node_metrics.sort_values(["betweenness", "StockCode"], ascending=[False, True]).head(10)
display(top_strength[metric_columns])
display(top_betweenness[metric_columns])

,StockCode,Description,invoice_count,community_id,degree,strength,betweenness
95,85099B,JUMBO BAG RED RETROSPOT,2089,1,94,27237,0.376417
98,85123A,WHITE HANGING HEART T-LIGHT HOLDER,2265,3,98,22551,0.230468
5,20725,LUNCH BAG RED RETROSPOT,1565,2,86,21481,0.071944
45,22423,REGENCY CAKESTAND 3 TIER,1988,3,86,18009,0.059988
34,22197,POPCORN HOLDER,1392,3,87,16867,0.004329
15,21212,PACK OF 72 RETROSPOT CAKE CASES,1320,3,80,16329,0.023706
24,21931,JUMBO STORAGE BAG SUKI,1184,1,69,16226,0.000000
7,20727,LUNCH BAG BLACK SKULL.,1273,2,66,15474,0.000412
40,22383,LUNCH BAG SUKI DESIGN,1284,2,64,15388,0.000825
4,20724,RED RETROSPOT CHARLOTTE BAG,1032,2,71,15340,0.000206


,StockCode,Description,invoice_count,community_id,degree,strength,betweenness
95,85099B,JUMBO BAG RED RETROSPOT,2089,1,94,27237,0.376417
98,85123A,WHITE HANGING HEART T-LIGHT HOLDER,2265,3,98,22551,0.230468
5,20725,LUNCH BAG RED RETROSPOT,1565,2,86,21481,0.071944
45,22423,REGENCY CAKESTAND 3 TIER,1988,3,86,18009,0.059988
15,21212,PACK OF 72 RETROSPOT CAKE CASES,1320,3,80,16329,0.023706
27,22086,PAPER CHAIN KIT 50'S CHRISTMAS,1160,5,48,7941,0.015461
61,22720,SET OF 3 CAKE TINS PANTRY DESIGN,1385,3,77,14343,0.010926
85,47566,PARTY BUNTING,1685,3,75,15242,0.008658
58,22697,GREEN REGENCY TEACUP AND SAUCER,1013,3,60,11716,0.004741
34,22197,POPCORN HOLDER,1392,3,87,16867,0.004329


Hai bảng tiếp theo cho các chênh lệch thứ hạng lớn nhất. rank_gap âm: strength nổi bật hơn betweenness; rank_gap dương: vai trò trung gian nổi bật hơn tổng mức mua chung. Dùng average ranks cho các giá trị bằng nhau; đây là so sánh vai trò tương đối, không phải ngưỡng phân loại thương mại.

In [6]:
role_columns = ["StockCode", "Description", "strength", "betweenness", "strength_rank", "betweenness_rank", "rank_gap"]
display(node_metrics.loc[node_metrics["rank_gap"] < 0].sort_values(["rank_gap", "StockCode"]).head(5)[role_columns])
display(node_metrics.loc[node_metrics["rank_gap"] > 0].sort_values(["rank_gap", "StockCode"], ascending=[False, True]).head(5)[role_columns])

,StockCode,Description,strength,betweenness,strength_rank,betweenness_rank,rank_gap
24,21931,JUMBO STORAGE BAG SUKI,16226,0.0,7.0,65.0,-58.0
68,22961,JAM MAKING SET PRINTED,14617,0.0,12.0,65.0,-53.0
44,22411,JUMBO SHOPPER VINTAGE RED PAISLEY,14556,0.0,13.0,65.0,-52.0
57,22666,RECIPE BOX PANTRY YELLOW DESIGN,14554,0.0,14.0,65.0,-51.0
43,22386,JUMBO BAG PINK POLKADOT,13800,0.0,16.0,65.0,-49.0


,StockCode,Description,strength,betweenness,strength_rank,betweenness_rank,rank_gap
84,23355,HOT WATER BOTTLE KEEP CALM,1328,0.000618,91.0,20.5,70.5
30,22112,CHOCOLATE HOT WATER BOTTLE,3536,0.003711,77.0,11.0,66.0
32,22139,RETROSPOT TEA SET CERAMIC 11 PC,3159,0.000412,80.0,24.0,56.0
64,22727,ALARM CLOCK BAKELIKE RED,6925,0.001443,55.0,13.0,42.0
27,22086,PAPER CHAIN KIT 50'S CHRISTMAS,7941,0.015461,46.0,6.0,40.0


## 5. Candidate Network Comparison

Giữ baseline và ba ứng viên Phase 2: Top 50 / 175, Top 75 / 140, Top 100 / 125. Chỉ lọc bảng pair counts đầy đủ trong scope đã lưu; thêm tất cả selected nodes để giữ cả isolates. Không đếm lại cặp từ giao dịch, không ghi GraphML.

Coverage lấy từ Phase 2 trước khi lọc cạnh. Retained weight có mẫu số là toàn bộ positive pair counts trong từng scope, vì vậy phần trăm giữa các scope không có cùng mẫu số. Các community IDs chỉ có ý nghĩa trong từng cấu hình. Giữ thứ tự node theo baseline, cùng resolution, seeds và cách tính centrality cho mọi graph để hạn chế khác biệt do thứ tự duyệt.

Nếu baseline đã trùng một ứng viên theo cả Top N và threshold, chỉ phân tích cấu hình đó một lần và giữ tên baseline cho các output.

Ứng viên vượt phạm vi selected products hoặc pair counts đã lưu được bỏ qua và ghi rõ lý do. Bảng so sánh và output chỉ chứa các cấu hình đủ dữ liệu; không suy đoán sản phẩm/cạnh thiếu hoặc chạy lại Phase 2.

In [7]:
graphs = {baseline_name: G}
metrics = {baseline_name: node_metrics}
run_tables = []
comparison_rows = []
scope_comparisons = pd.DataFrame(phase2["sensitivity_analysis"]["product_scope"]["comparisons"])
configurations = [(baseline_name, TOP_N_PRODUCTS, MIN_EDGE_WEIGHT)]
configurations += [(name, top_n, threshold) for name, top_n, threshold in [
    ("top50_t175", 50, 175), ("top75_t140", 75, 140), ("top100_t125", 100, 125),
] if (top_n, threshold) != (TOP_N_PRODUCTS, MIN_EDGE_WEIGHT)]
config_names = {}
skipped_candidates = []

for name, top_n, threshold in configurations:
    codes = set(products.head(top_n)["StockCode"])
    scope_available = len(products) >= top_n and codes.issubset(phase2["pair_count_cache"]["stock_codes"])
    if not scope_available and name != baseline_name:
        skipped_candidates.append({"config_name": name, "top_n": top_n, "threshold": threshold,
            "status": "skipped", "reason": "Selected products hoặc pair counts đã lưu không đủ phạm vi này."})
        continue
    assert scope_available, "Saved Phase 2 data do not cover the official baseline."
    scoped_pairs = pairs.loc[pairs["ProductA"].isin(codes) & pairs["ProductB"].isin(codes)]
    retained = scoped_pairs.loc[scoped_pairs["pair_count"] >= threshold]
    expected_edges = {(a, b): int(w) for a, b, w in retained.itertuples(index=False, name=None)}
    if name != baseline_name:
        graph = nx.Graph()
        graph.add_nodes_from(node for node in G if node in codes)
        graph.add_weighted_edges_from((a, b, w) for (a, b), w in expected_edges.items())
        graphs[name] = graph
    graph = graphs[name]
    actual_edges = {tuple(sorted((a, b))): d["weight"] for a, b, d in graph.edges(data=True)}
    assert set(graph) == codes and actual_edges == expected_edges, f"Nodes or edges do not match {name}."
    assert all(np.isfinite(d["weight"]) and d["weight"] >= threshold for _, _, d in graph.edges(data=True))
    if name == baseline_name:
        groups, runs, stability = communities, baseline_runs, baseline_stability
    else:
        groups = louvain_partition(graph, RANDOM_STATE)
        runs, stability = stability_analysis(graph, groups)
        metrics[name] = centrality_table(graph, groups)
    info = {"config_name": name, "top_n": top_n, "threshold": threshold, "random_state": RANDOM_STATE, "resolution": resolution}
    config_names[(top_n, threshold)] = name
    run_tables.append(runs.assign(**info))
    coverage = scope_comparisons.loc[(scope_comparisons["top_n"] == top_n) & (scope_comparisons["threshold"] == threshold)].iloc[0]
    tops = {}
    for metric in ["strength", "betweenness"]:
        leading = metrics[name].sort_values([metric, "StockCode"], ascending=[False, True]).head(10)
        tops[f"top10_{metric}"] = json.dumps(leading[["StockCode", "Description", metric]].to_dict("records"), ensure_ascii=False)
    comparison_rows.append({
        **info, "graph_role": "official_baseline" if name == baseline_name else "in_memory_candidate",
        "nodes": len(graph), "edges": graph.number_of_edges(), "density": nx.density(graph),
        "connected_components": nx.number_connected_components(graph), "isolated_nodes": nx.number_of_isolates(graph),
        "number_of_communities": len(groups), "weighted_modularity": nx.community.modularity(graph, groups, weight="weight", resolution=resolution),
        "community_sizes": json.dumps(sorted(map(len, groups), reverse=True)), **stability,
        "invoice_coverage_pct": coverage["invoice_coverage_pct"],
        "product_occurrence_coverage_pct": coverage["product_occurrence_coverage_pct"],
        "retained_weight_pct_within_scope": 100 * retained["pair_count"].sum() / scoped_pairs["pair_count"].sum(),
        **tops,
    })

community_stability = pd.concat(run_tables, ignore_index=True)
candidate_network_comparison = pd.DataFrame(comparison_rows)
if skipped_candidates:
    display(pd.DataFrame(skipped_candidates))

### Community assignments giữa baseline và Top 100 / 125

ARI dưới đây đo thay đổi phân hoạch khi đổi cấu hình mạng, giữ seed RANDOM_STATE và đối chiếu cùng tập StockCode. Đây là phép so sánh khác với mean/min pairwise ARI qua random seeds trên một graph. ARI cao trong từng graph không đảm bảo phân hoạch của hai graph giống nhau. Nếu baseline chính là Top 100 / 125, ARI bằng 1 là tự so sánh, không phải bằng chứng robustness độc lập.

Bảng cộng đồng của Top 100 / 125 dùng lại assignments đã tính. Mỗi nhóm hiển thị tối đa 5 sản phẩm đại diện theo invoice_count; nhóm ít hơn 5 sản phẩm hiển thị toàn bộ. Description là bằng chứng để diễn giải, không xác nhận nhóm có ý nghĩa kinh doanh.

ARI nhỏ hơn 1 cho thấy kết luận RQ1 còn phụ thuộc cấu hình cạnh; không thể dùng độ ổn định theo seed để khẳng định các nhóm bất biến giữa cấu hình. Nhận xét từng nhóm bên dưới dựa trên từ xuất hiện trong Description của toàn bộ thành viên, đếm mỗi từ tối đa một lần mỗi sản phẩm và bỏ các từ chức năng thông dụng. Từ chung chỉ là bằng chứng mô tả, không phải nhãn ngành hàng được xác minh. Đọc toàn bộ các sản phẩm của nhóm nhỏ ngay trong bảng đại diện trước khi diễn giải.

Nếu dữ liệu đã lưu không đủ Top 100 / 125, bỏ qua cả ARI giữa hai cấu hình và phần diễn giải ứng viên này. Các kết quả baseline và ứng viên còn đủ dữ liệu vẫn tiếp tục được phân tích và lưu.

In [8]:
target_name = config_names.get((100, 125))
configuration_ari = np.nan
candidate_network_comparison["community_ari_vs_baseline_same_nodes"] = np.nan
candidate_network_comparison.loc[candidate_network_comparison["config_name"] == baseline_name, "community_ari_vs_baseline_same_nodes"] = 1.0
candidate_network_comparison["community_profiles"] = pd.Series(np.nan, index=candidate_network_comparison.index, dtype=object)
if target_name is None:
    display(pd.DataFrame([{"analysis": "Top 100 / 125 community comparison and interpretation", "status": "skipped",
        "reason": "Không đủ dữ liệu Phase 2 để tái tạo Top 100 / 125; tiếp tục các cấu hình đã phân tích."}]))
else:
    baseline_assignments = metrics[baseline_name].set_index("StockCode")["community_id"]
    target_assignments = metrics[target_name].set_index("StockCode")["community_id"]
    assert set(baseline_assignments.index) == set(target_assignments.index), "Cross-configuration ARI requires the same product set."
    shared_codes = sorted(baseline_assignments.index)
    configuration_ari = adjusted_rand(baseline_assignments.loc[shared_codes].tolist(), target_assignments.loc[shared_codes].tolist())
    display(pd.DataFrame([{
        "baseline": baseline_name, "comparison": target_name, "seed": RANDOM_STATE,
        "products_compared": len(shared_codes), "ari_between_configurations": configuration_ari,
        "comparison_type": "same_configuration" if target_name == baseline_name else "between_configurations",
    }]))
    candidate_network_comparison.loc[candidate_network_comparison["config_name"] == target_name, "community_ari_vs_baseline_same_nodes"] = configuration_ari

    target_members = metrics[target_name].sort_values(["community_id", "invoice_count", "StockCode"], ascending=[True, False, True]).copy()
    target_members["community_size"] = target_members.groupby("community_id")["StockCode"].transform("size")
    representatives = target_members.groupby("community_id", sort=True).head(5)
    display(representatives[["community_id", "community_size", "StockCode", "Description", "invoice_count"]])
    profile_rows = []
    for community_id, members in target_members.groupby("community_id", sort=True):
        words = Counter()
        for tokens in members["Description"].str.findall(r"[A-Z]+"):
            words.update(set(tokens) - {"A", "AND", "FOR", "IN", "OF", "PACK", "PC", "PIECE", "S", "SET", "THE", "WITH"})
        frequent_words = sorted(words.items(), key=lambda item: (-item[1], item[0]))[:3]
        shared_terms = [word for word, count in frequent_words if count * 2 >= len(members)]
        comment = ("Ít nhất nửa nhóm nhắc " + ", ".join(shared_terms) + "; có nét chung trong mô tả."
                   if shared_terms else "Chưa thấy từ nội dung chung ở ít nhất nửa nhóm; cần đọc kỹ các sản phẩm khác nhau.")
        profile_rows.append({
            "community_id": int(community_id), "size": len(members),
            "representative_products": members.head(5)[["StockCode", "Description", "invoice_count"]].to_dict("records"),
            "description_evidence": "; ".join(f"{word}: {count}/{len(members)}" for word, count in frequent_words),
            "interpretation": comment,
        })
    target_community_profiles = pd.DataFrame(profile_rows)
    display(target_community_profiles[["community_id", "size", "description_evidence", "interpretation"]])
    candidate_network_comparison.loc[candidate_network_comparison["config_name"] == target_name, "community_profiles"] = json.dumps(profile_rows, ensure_ascii=False)

,baseline,comparison,seed,products_compared,ari_between_configurations,comparison_type
0,baseline_top100_t125,baseline_top100_t125,42,100,1.0,same_configuration


,community_id,community_size,StockCode,Description,invoice_count
95,1,18,85099B,JUMBO BAG RED RETROSPOT,2089
74,1,18,23203,JUMBO BAG VINTAGE DOILY,1231
43,1,18,22386,JUMBO BAG PINK POLKADOT,1218
24,1,18,21931,JUMBO STORAGE BAG SUKI,1184
44,1,18,22411,JUMBO SHOPPER VINTAGE RED PAISLEY,1175
5,2,21,20725,LUNCH BAG RED RETROSPOT,1565
40,2,21,22383,LUNCH BAG SUKI DESIGN,1284
7,2,21,20727,LUNCH BAG BLACK SKULL.,1273
39,2,21,22382,LUNCH BAG SPACEBOY DESIGN,1157
8,2,21,20728,LUNCH BAG CARS BLUE,1150


,community_id,size,description_evidence,interpretation
0,1,18,BAG: 16/18; JUMBO: 16/18; VINTAGE: 4/18,"Ít nhất nửa nhóm nhắc BAG, JUMBO; có nét chung..."
1,2,21,BAG: 16/21; LUNCH: 14/21; DESIGN: 6/21,"Ít nhất nửa nhóm nhắc BAG, LUNCH; có nét chung..."
2,3,44,HEART: 7/44; RETROSPOT: 7/44; LIGHT: 6/44,Chưa thấy từ nội dung chung ở ít nhất nửa nhóm...
3,4,4,SIGN: 4/4; METAL: 3/4; CHOCOLATE: 1/4,"Ít nhất nửa nhóm nhắc SIGN, METAL; có nét chun..."
4,5,13,ALARM: 3/13; BAKELIKE: 3/13; BOTTLE: 3/13,Chưa thấy từ nội dung chung ở ít nhất nửa nhóm...


Thứ hạng giữa hai mạng được tính lại trên tập StockCode chung, dùng Spearman (tương quan Pearson của average ranks). Strength và betweenness vẫn được tính trên toàn bộ từng graph, nên thay đổi thứ hạng phản ánh cả các node/cạnh ngoài tập chung. Bảng này không đánh giá các sản phẩm chỉ có ở scope lớn hơn. Top 10 hiển thị phá hòa bằng StockCode; các vị trí sát nhau hoặc bằng nhau không nên diễn giải quá mức.

In [9]:
ranking_rows = []
for name_a, name_b in combinations(graphs, 2):
    a = metrics[name_a].set_index("StockCode")
    b = metrics[name_b].set_index("StockCode")
    common = sorted(set(a.index) & set(b.index))
    row = {"config_a": name_a, "config_b": name_b, "common_nodes": len(common)}
    for metric in ["strength", "betweenness"]:
        rank_a = a.loc[common, metric].rank(ascending=False, method="average")
        rank_b = b.loc[common, metric].rank(ascending=False, method="average")
        row[f"{metric}_spearman"] = rank_a.corr(rank_b) if rank_a.nunique() > 1 and rank_b.nunique() > 1 else np.nan
    ranking_rows.append(row)
ranking_comparison = pd.DataFrame(ranking_rows, columns=[
    "config_a", "config_b", "common_nodes", "strength_spearman", "betweenness_spearman",
])
display(ranking_comparison.round(4))
for name in list(graphs)[1:]:
    display(metrics[name].sort_values(["strength", "StockCode"], ascending=[False, True]).head(10)[metric_columns].assign(config_name=name, ranking_by="strength"))
    display(metrics[name].sort_values(["betweenness", "StockCode"], ascending=[False, True]).head(10)[metric_columns].assign(config_name=name, ranking_by="betweenness"))

baseline_rankings = ranking_comparison.loc[ranking_comparison["config_a"] == baseline_name].rename(columns={"config_b": "config_name"})
candidate_network_comparison = candidate_network_comparison.merge(
    baseline_rankings[["config_name", "common_nodes", "strength_spearman", "betweenness_spearman"]].rename(columns={
        "common_nodes": "common_nodes_vs_baseline", "strength_spearman": "strength_spearman_vs_baseline", "betweenness_spearman": "betweenness_spearman_vs_baseline"}),
    on="config_name", how="left", validate="one_to_one",
)
candidate_network_comparison.loc[candidate_network_comparison["config_name"] == baseline_name,
    ["common_nodes_vs_baseline", "strength_spearman_vs_baseline", "betweenness_spearman_vs_baseline"]] = [len(G), 1.0, 1.0]

,config_a,config_b,common_nodes,strength_spearman,betweenness_spearman
0,baseline_top100_t125,top50_t175,50,0.9559,0.6718
1,baseline_top100_t125,top75_t140,75,0.9940,0.8123
2,top50_t175,top75_t140,50,0.9600,0.7127


,StockCode,Description,invoice_count,community_id,degree,strength,betweenness,config_name,ranking_by
47,85099B,JUMBO BAG RED RETROSPOT,2089,1,44,15300,0.386054,top50_t175,strength
49,85123A,WHITE HANGING HEART T-LIGHT HOLDER,2265,2,45,12432,0.202381,top50_t175,strength
1,20725,LUNCH BAG RED RETROSPOT,1565,1,37,12173,0.082483,top50_t175,strength
42,47566,PARTY BUNTING,1685,2,37,9011,0.016156,top50_t175,strength
20,22423,REGENCY CAKESTAND 3 TIER,1988,2,34,8926,0.055272,top50_t175,strength
16,22383,LUNCH BAG SUKI DESIGN,1284,1,25,8440,0.000850,top50_t175,strength
8,21931,JUMBO STORAGE BAG SUKI,1184,1,30,8438,0.000000,top50_t175,strength
14,22197,POPCORN HOLDER,1392,3,35,8223,0.007653,top50_t175,strength
3,20727,LUNCH BAG BLACK SKULL.,1273,1,24,8073,0.000000,top50_t175,strength
6,21212,PACK OF 72 RETROSPOT CAKE CASES,1320,2,31,7614,0.002551,top50_t175,strength


,StockCode,Description,invoice_count,community_id,degree,strength,betweenness,config_name,ranking_by
47,85099B,JUMBO BAG RED RETROSPOT,2089,1,44,15300,0.386054,top50_t175,betweenness
49,85123A,WHITE HANGING HEART T-LIGHT HOLDER,2265,2,45,12432,0.202381,top50_t175,betweenness
1,20725,LUNCH BAG RED RETROSPOT,1565,1,37,12173,0.082483,top50_t175,betweenness
20,22423,REGENCY CAKESTAND 3 TIER,1988,2,34,8926,0.055272,top50_t175,betweenness
12,22139,RETROSPOT TEA SET CERAMIC 11 PC,962,2,3,671,0.040816,top50_t175,betweenness
42,47566,PARTY BUNTING,1685,2,37,9011,0.016156,top50_t175,betweenness
14,22197,POPCORN HOLDER,1392,3,35,8223,0.007653,top50_t175,betweenness
6,21212,PACK OF 72 RETROSPOT CAKE CASES,1320,2,31,7614,0.002551,top50_t175,betweenness
15,22382,LUNCH BAG SPACEBOY DESIGN,1157,1,20,6470,0.001701,top50_t175,betweenness
32,22961,JAM MAKING SET PRINTED,1162,2,25,6170,0.001701,top50_t175,betweenness


,StockCode,Description,invoice_count,community_id,degree,strength,betweenness,config_name,ranking_by
71,85099B,JUMBO BAG RED RETROSPOT,2089,1,68,21942,0.358386,top75_t140,strength
74,85123A,WHITE HANGING HEART T-LIGHT HOLDER,2265,2,73,17951,0.208071,top75_t140,strength
3,20725,LUNCH BAG RED RETROSPOT,1565,1,64,17394,0.054424,top75_t140,strength
35,22423,REGENCY CAKESTAND 3 TIER,1988,2,64,14427,0.050722,top75_t140,strength
12,21212,PACK OF 72 RETROSPOT CAKE CASES,1320,2,61,13359,0.033691,top75_t140,strength
28,22197,POPCORN HOLDER,1392,2,63,13157,0.002962,top75_t140,strength
65,47566,PARTY BUNTING,1685,2,60,12960,0.011847,top75_t140,strength
19,21931,JUMBO STORAGE BAG SUKI,1184,1,50,12864,0.000000,top75_t140,strength
31,22383,LUNCH BAG SUKI DESIGN,1284,1,47,12345,0.000370,top75_t140,strength
34,22411,JUMBO SHOPPER VINTAGE RED PAISLEY,1175,1,53,12323,0.000000,top75_t140,strength


,StockCode,Description,invoice_count,community_id,degree,strength,betweenness,config_name,ranking_by
71,85099B,JUMBO BAG RED RETROSPOT,2089,1,68,21942,0.358386,top75_t140,betweenness
74,85123A,WHITE HANGING HEART T-LIGHT HOLDER,2265,2,73,17951,0.208071,top75_t140,betweenness
3,20725,LUNCH BAG RED RETROSPOT,1565,1,64,17394,0.054424,top75_t140,betweenness
35,22423,REGENCY CAKESTAND 3 TIER,1988,2,64,14427,0.050722,top75_t140,betweenness
12,21212,PACK OF 72 RETROSPOT CAKE CASES,1320,2,61,13359,0.033691,top75_t140,betweenness
22,22086,PAPER CHAIN KIT 50'S CHRISTMAS,1160,3,29,5293,0.022214,top75_t140,betweenness
24,22112,CHOCOLATE HOT WATER BOTTLE,858,3,13,2143,0.013328,top75_t140,betweenness
65,47566,PARTY BUNTING,1685,2,60,12960,0.011847,top75_t140,betweenness
45,22720,SET OF 3 CAKE TINS PANTRY DESIGN,1385,2,58,11641,0.009996,top75_t140,betweenness
63,23301,GARDENERS KNEELING PAD KEEP CALM,913,3,31,5199,0.009256,top75_t140,betweenness


### Community Quality — C3 và C5 của Top 100 / 125

C3/C5 là các nhóm của ứng viên Top 100 / 125 tại RANDOM_STATE, không phải nhãn tương đương giữa các lần phân nhóm. Xem 15 cặp có pair_count cao nhất trong mỗi nhóm, kèm StockCode và Description của hai đầu cạnh. Các cặp lấy từ bảng counts trước lọc; cột retained_at_125 chỉ rõ cặp được giữ trong graph.

Đánh giá nhóm bằng cả các quan hệ mua chung mạnh và liên kết ra ngoài. Sản phẩm khác loại vẫn có thể cùng xuất hiện trong một basket hợp lý; sự giống nhau của Description không phải điều kiện bắt buộc của một cộng đồng.

In [10]:
focus_groups = {}
focus_top_pairs = pd.DataFrame()
if target_name is None:
    display(pd.DataFrame([{"analysis": "C3/C5 quality", "status": "skipped", "reason": "Top 100 / 125 chưa đủ dữ liệu."}]))
else:
    target_graph = graphs[target_name]
    focus_groups = {community_id: set(target_assignments.index[target_assignments.eq(community_id)])
                    for community_id in [3, 5] if target_assignments.eq(community_id).any()}
    descriptions = products.set_index("StockCode")["Description"]
    pair_tables = []
    for community_id, members in focus_groups.items():
        strongest = pairs.loc[pairs["ProductA"].isin(members) & pairs["ProductB"].isin(members)].sort_values(
            ["pair_count", "ProductA", "ProductB"], ascending=[False, True, True]).head(15).copy()
        strongest = strongest.rename(columns={"ProductA": "StockCodeA", "ProductB": "StockCodeB"})
        strongest["DescriptionA"] = strongest["StockCodeA"].map(descriptions)
        strongest["DescriptionB"] = strongest["StockCodeB"].map(descriptions)
        strongest["community_id"] = community_id
        strongest["retained_at_125"] = strongest["pair_count"].ge(125)
        strongest = strongest[["community_id", "StockCodeA", "DescriptionA", "StockCodeB", "DescriptionB", "pair_count", "retained_at_125"]]
        pair_tables.append(strongest)
        display(strongest)
    focus_top_pairs = pd.concat(pair_tables, ignore_index=True) if pair_tables else pd.DataFrame()

,community_id,StockCodeA,DescriptionA,StockCodeB,DescriptionB,pair_count,retained_at_125
4090,3,22697,GREEN REGENCY TEACUP AND SAUCER,22699,ROSES REGENCY TEACUP AND SAUCER,767,True
4089,3,22697,GREEN REGENCY TEACUP AND SAUCER,22698,PINK REGENCY TEACUP AND SAUCER,632,True
4130,3,22698,PINK REGENCY TEACUP AND SAUCER,22699,ROSES REGENCY TEACUP AND SAUCER,598,True
4860,3,82482,WOODEN PICTURE FRAME WHITE FINISH,82494L,WOODEN FRAME ANTIQUE WHITE,539,True
3479,3,22423,REGENCY CAKESTAND 3 TIER,22699,ROSES REGENCY TEACUP AND SAUCER,525,True
3477,3,22423,REGENCY CAKESTAND 3 TIER,22697,GREEN REGENCY TEACUP AND SAUCER,509,True
1708,3,21733,RED HANGING HEART T-LIGHT HOLDER,85123A,WHITE HANGING HEART T-LIGHT HOLDER,497,True
3624,3,22469,HEART OF WICKER SMALL,22470,HEART OF WICKER LARGE,493,True
4764,3,23298,SPOTTY BUNTING,47566,PARTY BUNTING,483,True
4422,3,22960,JAM MAKING SET WITH JARS,22961,JAM MAKING SET PRINTED,472,True


,community_id,StockCodeA,DescriptionA,StockCodeB,DescriptionB,pair_count,retained_at_125
4284,5,22726,ALARM CLOCK BAKELIKE GREEN,22727,ALARM CLOCK BAKELIKE RED,640,True
2360,5,22086,PAPER CHAIN KIT 50'S CHRISTMAS,22910,PAPER CHAIN KIT VINTAGE CHRISTMAS,552,True
4779,5,23300,GARDENERS KNEELING PAD CUP OF TEA,23301,GARDENERS KNEELING PAD KEEP CALM,546,True
4320,5,22727,ALARM CLOCK BAKELIKE RED,22728,ALARM CLOCK BAKELIKE PINK,470,True
4285,5,22726,ALARM CLOCK BAKELIKE GREEN,22728,ALARM CLOCK BAKELIKE PINK,418,True
2465,5,22111,SCOTTIE DOG HOT WATER BOTTLE,22112,CHOCOLATE HOT WATER BOTTLE,311,True
2588,5,22112,CHOCOLATE HOT WATER BOTTLE,23355,HOT WATER BOTTLE KEEP CALM,269,True
2519,5,22111,SCOTTIE DOG HOT WATER BOTTLE,23355,HOT WATER BOTTLE KEEP CALM,197,True
2357,5,22086,PAPER CHAIN KIT 50'S CHRISTMAS,22726,ALARM CLOCK BAKELIKE GREEN,185,True
2364,5,22086,PAPER CHAIN KIT 50'S CHRISTMAS,23084,RABBIT NIGHT LIGHT,185,True


### Large Invoice Evidence — Phase 2

Bảng dưới giữ nguyên kết quả sensitivity của Phase 2: loại hóa đơn có ít nhất 100 sản phẩm phân biệt trong toàn bộ basket, trước khi lọc Top N. Phase 2 chưa lưu pair counts riêng của scenario này. Phần robustness tiếp theo dùng ngoại lệ được cho phép: đọc invoice_items đã chuẩn bị ở Phase 1, kiểm tra checksum khớp input Phase 2 và chỉ đếm lại cặp cho scenario. Không cleaning, thay đổi selected products hoặc sửa dữ liệu đã lưu.

In [11]:
large_invoice = phase2["sensitivity_analysis"]["large_invoices"]
display(pd.Series({key: large_invoice[key] for key in ["excluded_invoices_in_scenario", "large_invoice_selected_product_event_pct"]}, name="Phase 2 large invoice contribution"))
display(pd.DataFrame(large_invoice["threshold_comparisons"])[["threshold", "case", "edges", "density", "connected_components", "isolated_nodes", "edge_weight_sum", "edge_weight_vs_baseline_pct"]])

excluded_invoices_in_scenario               636.000000
large_invoice_selected_product_event_pct     49.440924
Name: Phase 2 large invoice contribution, dtype: float64

,threshold,case,edges,density,connected_components,isolated_nodes,edge_weight_sum,edge_weight_vs_baseline_pct
0,100,baseline,2976,0.601212,1,0,506431,100.000000
1,100,without_large_invoices,728,0.147071,4,2,131969,26.058634
2,140,baseline,1606,0.324444,1,0,343840,100.000000
3,140,without_large_invoices,402,0.081212,18,13,93797,27.279258
4,150,baseline,1375,0.277778,1,0,310402,100.000000
5,150,without_large_invoices,354,0.071515,20,15,86846,27.978557


### Large Invoice Robustness — Top 100 / 125

Baseline ở phần này là ứng viên Top 100 / 125 giữ mọi hóa đơn hợp lệ, phân biệt với graph chính thức theo config. Sensitivity giữ nguyên 100 sản phẩm đã chọn và threshold 125, chỉ bỏ các hóa đơn có ≥100 StockCode phân biệt trong toàn bộ basket. Baseline dùng graph và Louvain đã tính; sensitivity dùng cùng seed, resolution và trọng số tần suất. Giữ cả isolated nodes.

ARI đo thay đổi phân hoạch trên cùng nodes. Bảng overlap cho biết nhóm bị tách/gộp. Bảng sản phẩm tính số bạn cùng nhóm bị mất hoặc được thêm, độc lập với community IDs: đổi số nhãn đơn thuần không bị xem là đổi nhóm. Hiển thị tối đa 5 sản phẩm có thay đổi quan hệ cùng nhóm nhiều nhất từ mỗi nhóm C3/C5 gốc.

In [12]:
robustness_ari = np.nan
scenario_summary = pd.DataFrame()
quality_scenarios = []
if target_name is not None:
    quality_scenarios = [("all_valid_invoices", target_graph, "pair_count")]
    scenario_pair_counts = pairs.loc[pairs["ProductA"].isin(target_graph) & pairs["ProductB"].isin(target_graph)].copy()
    invoice_info = phase2["input_files"].get("invoice_items", {})
    invoice_path = project_root / invoice_info.get("path", "")
    source_matches = invoice_path.is_file() and hashlib.sha256(invoice_path.read_bytes()).hexdigest() == invoice_info.get("sha256")
    if not source_matches:
        display(pd.DataFrame([{"analysis": "Large invoice robustness", "status": "skipped",
            "reason": "Thiếu invoice_items đã dùng ở Phase 2 hoặc checksum không khớp; không suy đoán scenario."}]))
    else:
        invoice_items = pd.read_parquet(invoice_path, columns=["InvoiceNo", "StockCode"])
        full_basket_sizes = invoice_items.groupby("InvoiceNo")["StockCode"].nunique()
        excluded_invoices = set(full_basket_sizes.index[full_basket_sizes.ge(100)])
        scenario_items = invoice_items.loc[~invoice_items["InvoiceNo"].isin(excluded_invoices) & invoice_items["StockCode"].isin(target_graph)]
        sensitivity_counter = Counter()
        for _, basket in scenario_items.groupby("InvoiceNo")["StockCode"]:
            sensitivity_counter.update(combinations(sorted(basket), 2))
        scenario_pair_counts["sensitivity_pair_count"] = [sensitivity_counter[(a, b)] for a, b in scenario_pair_counts[["ProductA", "ProductB"]].itertuples(index=False, name=None)]
        assert scenario_pair_counts["sensitivity_pair_count"].between(0, scenario_pair_counts["pair_count"]).all(), "Sensitivity counts must not exceed saved baseline counts."
        sensitivity_graph = nx.Graph()
        sensitivity_graph.add_nodes_from(target_graph.nodes(data=True))
        sensitivity_graph.add_weighted_edges_from(
            (a, b, int(w)) for a, b, w in scenario_pair_counts.loc[scenario_pair_counts["sensitivity_pair_count"].ge(125),
                ["ProductA", "ProductB", "sensitivity_pair_count"]].itertuples(index=False, name=None))
        assert set(sensitivity_graph) == set(target_graph), "Keep the same selected nodes, including isolates."
        sensitivity_groups = louvain_partition(sensitivity_graph, RANDOM_STATE)
        sensitivity_assignments = pd.Series(membership(sensitivity_groups), name="sensitivity_community")
        sensitivity_modularity = nx.community.modularity(sensitivity_graph, sensitivity_groups, weight="weight", resolution=resolution)
        robustness_ari = adjusted_rand(target_assignments.loc[shared_codes].tolist(), sensitivity_assignments.loc[shared_codes].tolist())
        assert np.isfinite([sensitivity_modularity, robustness_ari]).all()
        quality_scenarios.append(("without_large_invoices", sensitivity_graph, "sensitivity_pair_count"))
        target_result = candidate_network_comparison.loc[candidate_network_comparison["config_name"] == target_name].iloc[0]
        scenario_summary = pd.DataFrame([
            {"scenario": "all_valid_invoices", "invoices": len(full_basket_sizes), "excluded_invoices": 0,
             "nodes": len(target_graph), "edges": target_graph.number_of_edges(), "components": nx.number_connected_components(target_graph),
             "isolates": nx.number_of_isolates(target_graph), "communities": target_assignments.nunique(),
             "weighted_modularity": target_result["weighted_modularity"], "community_sizes": sorted(target_assignments.value_counts().tolist(), reverse=True), "ari_vs_all_invoices": 1.0},
            {"scenario": "without_large_invoices", "invoices": len(full_basket_sizes) - len(excluded_invoices), "excluded_invoices": len(excluded_invoices),
             "nodes": len(sensitivity_graph), "edges": sensitivity_graph.number_of_edges(), "components": nx.number_connected_components(sensitivity_graph),
             "isolates": nx.number_of_isolates(sensitivity_graph), "communities": len(sensitivity_groups),
             "weighted_modularity": sensitivity_modularity, "community_sizes": sorted(map(len, sensitivity_groups), reverse=True), "ari_vs_all_invoices": robustness_ari},
        ])
        with pd.option_context("display.max_colwidth", None):
            display(scenario_summary)
        assignment_changes = target_assignments.rename("baseline_community").to_frame().join(sensitivity_assignments).rename_axis("StockCode").reset_index()
        overlap = assignment_changes.groupby(["baseline_community", "sensitivity_community"])["StockCode"].transform("size")
        assignment_changes["lost_community_peers"] = assignment_changes.groupby("baseline_community")["StockCode"].transform("size") - overlap
        assignment_changes["gained_community_peers"] = assignment_changes.groupby("sensitivity_community")["StockCode"].transform("size") - overlap
        assignment_changes["changed_peers"] = assignment_changes["lost_community_peers"] + assignment_changes["gained_community_peers"]
        assignment_changes = assignment_changes.merge(products[["StockCode", "Description"]], on="StockCode", validate="one_to_one")
        display(pd.crosstab(assignment_changes["baseline_community"], assignment_changes["sensitivity_community"]))
        notable_changes = assignment_changes.loc[assignment_changes["baseline_community"].isin(focus_groups) & assignment_changes["changed_peers"].gt(0)].sort_values(
            ["baseline_community", "changed_peers", "StockCode"], ascending=[True, False, True]).groupby("baseline_community").head(5)
        display(notable_changes[["StockCode", "Description", "baseline_community", "sensitivity_community", "lost_community_peers", "gained_community_peers"]])

,scenario,invoices,excluded_invoices,nodes,edges,components,isolates,communities,weighted_modularity,community_sizes,ari_vs_all_invoices
0,all_valid_invoices,19773,0,100,2074,1,0,5,0.184622,"[44, 21, 18, 13, 4]",1.000000
1,without_large_invoices,19137,636,100,497,11,7,16,0.422710,"[23, 18, 17, 17, 9, 3, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1]",0.451329


sensitivity_community,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
baseline_community,,,,,,,,,,,,,,,,
1,17,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0
2,0,17,0,0,0,0,0,0,0,0,1,0,3,0,0,0
3,0,0,14,20,1,1,1,1,2,0,0,1,0,0,2,1
4,0,0,4,0,0,0,0,0,0,0,0,0,0,0,0,0
5,0,0,0,3,0,1,0,7,0,1,0,0,0,1,0,0


,StockCode,Description,baseline_community,sensitivity_community,lost_community_peers,gained_community_peers
34,22197,POPCORN HOLDER,3,8,43,8
17,21731,RED TOADSTOOL LED NIGHT LIGHT,3,6,43,1
16,21232,STRAWBERRY CERAMIC TRINKET BOX,3,5,43,0
20,21790,VINTAGE SNAP CARDS,3,7,43,0
51,22558,CLOTHES PEGS RETROSPOT PACK 24,3,12,43,0
63,22726,ALARM CLOCK BAKELIKE GREEN,5,4,10,20
64,22727,ALARM CLOCK BAKELIKE RED,5,4,10,20
65,22728,ALARM CLOCK BAKELIKE PINK,5,4,10,20
70,23084,RABBIT NIGHT LIGHT,5,6,12,1
47,22467,GUMBALL COAT RACK,5,10,12,0


### Internal Links — C3/C5 giữ cố định thành viên baseline

Để so sánh cùng một nhóm sản phẩm, giữ nguyên thành viên C3/C5 của Top 100 / 125 khi đo cả hai scenario, kể cả khi Louvain tách/gộp chúng. internal_weight cộng mỗi cạnh nội bộ một lần; external_weight cộng các cạnh nối nhóm ra ngoài. internal_strength_share = 2 × internal_weight / (2 × internal_weight + external_weight), biểu diễn tỷ lệ strength của nhóm dành cho liên kết nội bộ.

Phân biệt mất tần suất mua chung với mất cạnh do threshold: all_positive_internal_weight dùng pair counts chưa lọc; internal_edges và internal_weight chỉ dùng cạnh ≥125. Bảng các cặp mạnh đối chiếu lại đúng các cặp baseline đã chọn, không chọn cặp mới để làm kết quả trông ổn định hơn.

In [13]:
if quality_scenarios:
    quality_rows = []
    for scenario, graph, count_column in quality_scenarios:
        for community_id, members in focus_groups.items():
            internal_edges = list(graph.subgraph(members).edges(data=True))
            external_edges = [(a, b, d) for a, b, d in graph.edges(data=True) if (a in members) != (b in members)]
            internal_weight = sum(d["weight"] for _, _, d in internal_edges)
            external_weight = sum(d["weight"] for _, _, d in external_edges)
            within = scenario_pair_counts["ProductA"].isin(members) & scenario_pair_counts["ProductB"].isin(members)
            full_weight = int(scenario_pair_counts.loc[within, count_column].sum())
            full_baseline_weight = int(scenario_pair_counts.loc[within, "pair_count"].sum())
            baseline_internal = target_graph.subgraph(members)
            surviving_edges = sum(graph.has_edge(a, b) for a, b in baseline_internal.edges())
            quality_rows.append({"scenario": scenario, "baseline_community": community_id, "products": len(members),
                "internal_edges": len(internal_edges), "external_edges": len(external_edges),
                "internal_weight": internal_weight, "external_weight": external_weight,
                "internal_strength_share": 2 * internal_weight / (2 * internal_weight + external_weight) if internal_weight or external_weight else np.nan,
                "all_positive_internal_weight": full_weight,
                "all_positive_internal_weight_retained_pct": 100 * full_weight / full_baseline_weight if full_baseline_weight else np.nan,
                "baseline_internal_edges_surviving_pct": 100 * surviving_edges / baseline_internal.number_of_edges() if baseline_internal.number_of_edges() else np.nan})
    community_quality_robustness = pd.DataFrame(quality_rows)
    display(community_quality_robustness.round(4))
    if len(quality_scenarios) == 2 and not focus_top_pairs.empty:
        strong_pair_changes = focus_top_pairs.copy()
        strong_pair_changes["sensitivity_pair_count"] = [sensitivity_counter[(a, b)] for a, b in strong_pair_changes[["StockCodeA", "StockCodeB"]].itertuples(index=False, name=None)]
        strong_pair_changes["pair_weight_retained_pct"] = 100 * strong_pair_changes["sensitivity_pair_count"] / strong_pair_changes["pair_count"]
        strong_pair_changes["sensitivity_edge_retained_at_125"] = strong_pair_changes["sensitivity_pair_count"].ge(125)
        for _, table in strong_pair_changes.groupby("community_id"):
            display(table.drop(columns="retained_at_125"))

,scenario,baseline_community,products,internal_edges,external_edges,internal_weight,external_weight,internal_strength_share,all_positive_internal_weight,all_positive_internal_weight_retained_pct,baseline_internal_edges_surviving_pct
0,all_valid_invoices,3,44,523,815,100940,136709,0.5962,138481,100.0000,100.0000
1,all_valid_invoices,5,13,37,242,7726,37280,0.2930,11031,100.0000,100.0000
2,without_large_invoices,3,44,81,74,17048,11455,0.7485,66020,47.6744,15.4876
3,without_large_invoices,5,13,10,12,3083,1751,0.7788,7106,64.4185,27.0270


,community_id,StockCodeA,DescriptionA,StockCodeB,DescriptionB,pair_count,sensitivity_pair_count,pair_weight_retained_pct,sensitivity_edge_retained_at_125
0,3,22697,GREEN REGENCY TEACUP AND SAUCER,22699,ROSES REGENCY TEACUP AND SAUCER,767,550,71.707953,True
1,3,22697,GREEN REGENCY TEACUP AND SAUCER,22698,PINK REGENCY TEACUP AND SAUCER,632,468,74.050633,True
2,3,22698,PINK REGENCY TEACUP AND SAUCER,22699,ROSES REGENCY TEACUP AND SAUCER,598,441,73.745819,True
3,3,82482,WOODEN PICTURE FRAME WHITE FINISH,82494L,WOODEN FRAME ANTIQUE WHITE,539,457,84.786642,True
4,3,22423,REGENCY CAKESTAND 3 TIER,22699,ROSES REGENCY TEACUP AND SAUCER,525,399,76.000000,True
5,3,22423,REGENCY CAKESTAND 3 TIER,22697,GREEN REGENCY TEACUP AND SAUCER,509,359,70.530452,True
6,3,21733,RED HANGING HEART T-LIGHT HOLDER,85123A,WHITE HANGING HEART T-LIGHT HOLDER,497,410,82.494970,True
7,3,22469,HEART OF WICKER SMALL,22470,HEART OF WICKER LARGE,493,401,81.338742,True
8,3,23298,SPOTTY BUNTING,47566,PARTY BUNTING,483,408,84.472050,True
9,3,22960,JAM MAKING SET WITH JARS,22961,JAM MAKING SET PRINTED,472,294,62.288136,True


,community_id,StockCodeA,DescriptionA,StockCodeB,DescriptionB,pair_count,sensitivity_pair_count,pair_weight_retained_pct,sensitivity_edge_retained_at_125
15,5,22726,ALARM CLOCK BAKELIKE GREEN,22727,ALARM CLOCK BAKELIKE RED,640,560,87.500000,True
16,5,22086,PAPER CHAIN KIT 50'S CHRISTMAS,22910,PAPER CHAIN KIT VINTAGE CHRISTMAS,552,436,78.985507,True
17,5,23300,GARDENERS KNEELING PAD CUP OF TEA,23301,GARDENERS KNEELING PAD KEEP CALM,546,416,76.190476,True
18,5,22727,ALARM CLOCK BAKELIKE RED,22728,ALARM CLOCK BAKELIKE PINK,470,398,84.680851,True
19,5,22726,ALARM CLOCK BAKELIKE GREEN,22728,ALARM CLOCK BAKELIKE PINK,418,344,82.296651,True
20,5,22111,SCOTTIE DOG HOT WATER BOTTLE,22112,CHOCOLATE HOT WATER BOTTLE,311,259,83.279743,True
21,5,22112,CHOCOLATE HOT WATER BOTTLE,23355,HOT WATER BOTTLE KEEP CALM,269,240,89.219331,True
22,5,22111,SCOTTIE DOG HOT WATER BOTTLE,23355,HOT WATER BOTTLE KEEP CALM,197,178,90.355330,True
23,5,22086,PAPER CHAIN KIT 50'S CHRISTMAS,22726,ALARM CLOCK BAKELIKE GREEN,185,92,49.729730,False
24,5,22086,PAPER CHAIN KIT 50'S CHRISTMAS,23084,RABBIT NIGHT LIGHT,185,102,55.135135,False


### Robustness Findings

Bảng dưới phân biệt mức ổn định theo seed, lựa chọn cạnh và thành phần hóa đơn. Mean/min ARI của nhiều seeds là thống kê khác với một ARI giữa hai graph. Với C3/C5, đối chiếu cặp mạnh cục bộ với sự tách/gộp của cả nhóm: một cộng đồng có thể có những nhóm con mua chung hợp lý dù ranh giới tổng thể nhạy với hóa đơn lớn.

Các cặp đại diện được lấy trực tiếp từ bảng cặp mạnh; đây là bằng chứng về số lần cùng xuất hiện, không chứng minh quan hệ nhân quả. Trọng số hoặc internal_strength_share tăng tương đối sau lọc cũng có thể do các cạnh ngoài nhóm bị loại mạnh hơn. Đánh giá độ tin cậy của Top 100 / 125 cần giữ cả những quan hệ còn bền và giới hạn của phân hoạch; chưa đủ để coi đây là cấu hình tối ưu hoặc đề xuất sản phẩm cross-selling.

In [14]:
if len(quality_scenarios) == 2:
    robustness_levels = pd.DataFrame([
        {"comparison": "Random seeds — Top 100/125", "measure": "mean pairwise ARI", "ARI": target_result["mean_pairwise_ari"]},
        {"comparison": "Random seeds — Top 100/125", "measure": "min pairwise ARI", "ARI": target_result["min_pairwise_ari"]},
        {"comparison": f"{baseline_name} vs {target_name}", "measure": "between configurations" if baseline_name != target_name else "self comparison", "ARI": configuration_ari},
        {"comparison": "All invoices vs without large invoices — Top 100/125", "measure": "between invoice scenarios", "ARI": robustness_ari},
    ])
    display(robustness_levels)
    focus_findings = []
    for community_id, table in strong_pair_changes.groupby("community_id"):
        members = focus_groups[community_id]
        retained_pairs = int(table["sensitivity_edge_retained_at_125"].sum())
        evidence = community_quality_robustness.loc[(community_quality_robustness["scenario"] == "without_large_invoices") & (community_quality_robustness["baseline_community"] == community_id)].iloc[0]
        focus_findings.append({"baseline_community": community_id,
            "strong_pair_examples": "; ".join(f"{a} ↔ {b}" for a, b in table[["DescriptionA", "DescriptionB"]].head(3).itertuples(index=False, name=None)),
            "strong_pairs_checked": len(table), "strong_pairs_still_at_125": retained_pairs,
            "strong_pair_weight_retained_pct": 100 * table["sensitivity_pair_count"].sum() / table["pair_count"].sum(),
            "all_internal_pair_weight_retained_pct": evidence["all_positive_internal_weight_retained_pct"],
            "baseline_internal_edges_surviving_pct": evidence["baseline_internal_edges_surviving_pct"],
            "sensitivity_communities_spanned": sensitivity_assignments.loc[sorted(members)].nunique(),
            "finding": "Các cặp mạnh nhất còn tồn tại; không đồng nghĩa toàn bộ nhóm giữ nguyên." if retained_pairs == len(table) else "Một số cặp rơi dưới threshold; cần phân biệt quan hệ còn bền với liên kết nhạy với hóa đơn lớn.",
        })
    with pd.option_context("display.max_colwidth", None):
        display(pd.DataFrame(focus_findings).round(4))

,comparison,measure,ARI
0,Random seeds — Top 100/125,mean pairwise ARI,0.940358
1,Random seeds — Top 100/125,min pairwise ARI,0.796008
2,baseline_top100_t125 vs baseline_top100_t125,self comparison,1.000000
3,All invoices vs without large invoices — Top 1...,between invoice scenarios,0.451329


,baseline_community,strong_pair_examples,strong_pairs_checked,strong_pairs_still_at_125,strong_pair_weight_retained_pct,all_internal_pair_weight_retained_pct,baseline_internal_edges_surviving_pct,sensitivity_communities_spanned,finding
0,3,GREEN REGENCY TEACUP AND SAUCER ↔ ROSES REGENCY TEACUP AND SAUCER; GREEN REGENCY TEACUP AND SAUCER ↔ PINK REGENCY TEACUP AND SAUCER; PINK REGENCY TEACUP AND SAUCER ↔ ROSES REGENCY TEACUP AND SAUCER,15,15,73.7194,47.6744,15.4876,10,Các cặp mạnh nhất còn tồn tại; không đồng nghĩa toàn bộ nhóm giữ nguyên.
1,5,ALARM CLOCK BAKELIKE GREEN ↔ ALARM CLOCK BAKELIKE RED; PAPER CHAIN KIT 50'S CHRISTMAS ↔ PAPER CHAIN KIT VINTAGE CHRISTMAS; GARDENERS KNEELING PAD CUP OF TEA ↔ GARDENERS KNEELING PAD KEEP CALM,15,9,75.7511,64.4185,27.0270,5,Một số cặp rơi dưới threshold; cần phân biệt quan hệ còn bền với liên kết nhạy với hóa đơn lớn.


## 6. Validation & Research Findings

Các assert đặt gần bước tính bảo vệ: graph undirected với trọng số dương hữu hạn, mỗi node thuộc đúng một cộng đồng, modularity/centrality hợp lệ, cùng seed cho cùng phân hoạch và variants khớp selected nodes/retained pairs. Không chạy lại kiểm tra preprocessing hoặc đếm cặp baseline Phase 2. Sensitivity là ngoại lệ: chỉ đếm cặp sau loại hóa đơn lớn từ invoice_items có checksum khớp Phase 2; kiểm tra counts không tăng và giữ cùng selected nodes.

Đọc kết quả RQ1 cùng mức ổn định và nội dung sản phẩm; đọc RQ2 bằng cả strength lẫn weighted betweenness. Centrality không đảm bảo doanh thu, quan hệ nhân quả hay hiệu quả cross-selling. Ngưỡng cạnh và scope có thể làm thay đổi kết luận; ARI theo seed chưa kiểm chứng độ bền khi thay đổi hóa đơn.

Đánh giá các ứng viên bằng cả weighted modularity, stability theo seed, ARI giữa cấu hình, coverage và nội dung sản phẩm. Modularity cao hơn không tự chứng minh cấu hình tốt hơn; ARI giữa hai cấu hình thấp hơn cho thấy kết luận RQ1 phụ thuộc lựa chọn cạnh ngay cả khi mỗi graph ổn định theo seed.

Các bảng dưới dùng biến của lần chạy hiện tại. Các mô tả cộng đồng dựa trên Description cần được đối chiếu với toàn bộ thành viên, đặc biệt nhóm nhỏ; chưa thể suy ra doanh thu hay hiệu quả cross-selling. Không áp dụng cấu hình tự động. Nếu người dùng chọn cấu hình khác baseline, cần cập nhật config và chạy lại Phase 2 → Phase 3 trước khi chốt kết quả nghiên cứu.

Để cân nhắc Top 100 / 125 cho nghiên cứu chính thức, đối chiếu khả năng giữ scope rộng, retained weight và ARI theo seed với ARI giữa cấu hình cùng bằng chứng Description của từng nhóm. Nhóm có từ mô tả chung có thể dễ diễn giải hơn; nhóm nhiều sản phẩm khác loại vẫn cần thận trọng. Top 50 / 175 và Top 75 / 140 là các phương án scope nhỏ hơn để đối chiếu sự đánh đổi coverage và độ ổn định; không xếp hạng cấu hình chỉ bằng modularity.

Chỉ đánh giá các cấu hình có kết quả trong bảng. Ứng viên bị bỏ qua vì thiếu dữ liệu chưa có cơ sở để kết luận.

Phân biệt ba loại độ bền: ARI qua seeds đo ngẫu nhiên của Louvain trên cùng graph; ARI giữa thresholds đo phụ thuộc lựa chọn cạnh; ARI giữa large-invoice scenarios đo phụ thuộc thành phần hóa đơn. Một chỉ số cao không thay thế hai kiểm tra còn lại. C3/C5 cần được đánh giá từ các cặp mạnh còn tồn tại, mức giữ trọng số và sự tách/gộp nhóm, không chỉ từ tên sản phẩm. Modularity tăng ở graph thưa hoặc phân mảnh không tự chứng minh chất lượng tốt hơn. Không đề xuất sản phẩm cross-selling từ kiểm tra này.

In [15]:
evaluation_columns = ["config_name", "nodes", "edges", "density", "number_of_communities", "weighted_modularity",
    "mean_pairwise_ari", "min_pairwise_ari", "community_ari_vs_baseline_same_nodes",
    "invoice_coverage_pct", "product_occurrence_coverage_pct", "retained_weight_pct_within_scope"]
display(candidate_network_comparison[evaluation_columns].round(4))

,config_name,nodes,edges,density,number_of_communities,weighted_modularity,mean_pairwise_ari,min_pairwise_ari,community_ari_vs_baseline_same_nodes,invoice_coverage_pct,product_occurrence_coverage_pct,retained_weight_pct_within_scope
0,baseline_top100_t125,100,2074,0.4190,5,0.1846,0.9404,0.7960,1.0,80.6251,18.8598,62.3659
1,top50_t175,50,474,0.3869,3,0.2338,0.8519,0.6228,NaN,71.3397,11.3589,57.1407
2,top75_t140,75,1242,0.4476,3,0.1860,0.8072,0.5478,NaN,77.3782,15.3953,63.6444


## 7. Outputs & Phase 3 Summary

Lưu bốn CSV trong outputs/tables. node_metrics và community_summary thuộc graph chính thức; community_stability và candidate_network_comparison chứa baseline và các ứng viên đủ dữ liệu, được phân biệt bằng config_name, top_n và threshold. Cột community_ari_vs_baseline_same_nodes ghi ARI giữa các cấu hình có cùng tập node đã được đối chiếu; các scope khác để trống. community_profiles lưu bằng chứng diễn giải của Top 100 / 125 khi ứng viên này đủ dữ liệu; nếu không, cột để trống. random_state và resolution ghi cấu hình phân tích; seed trong bảng stability chỉ rõ từng lần chạy.

Chưa thay đổi graph hoặc config chính thức. Nếu chọn một ứng viên cho kết quả nghiên cứu cuối cùng, cần được phép cập nhật config rồi chạy lại Phase 2 và Phase 3. Các notebook 04–05 sử dụng kết quả tương ứng với cấu hình chính thức đã chốt.

Các bảng Community Quality & Large Invoice Robustness được lưu trong output của notebook, không tạo thêm CSV và không thay đổi ý nghĩa của bốn CSV hiện có.

In [16]:
outputs = {
    "node_metrics.csv": node_metrics,
    "community_summary.csv": community_summary,
    "community_stability.csv": community_stability,
    "candidate_network_comparison.csv": candidate_network_comparison,
}
for filename, table in outputs.items():
    path = TABLES_DIR / filename
    table.to_csv(path, index=False)
    saved = pd.read_csv(path, dtype={"StockCode": "string"} if "StockCode" in table else None)
    pd.testing.assert_frame_equal(table.reset_index(drop=True), saved, check_dtype=False)
display(pd.DataFrame([{"file": str(TABLES_DIR / name), "rows": len(table), "status": "saved and read back"} for name, table in outputs.items()]))

,file,rows,status
0,/Users/trantuananh/co-purchase-network-finalte...,100,saved and read back
1,/Users/trantuananh/co-purchase-network-finalte...,5,saved and read back
2,/Users/trantuananh/co-purchase-network-finalte...,30,saved and read back
3,/Users/trantuananh/co-purchase-network-finalte...,3,saved and read back
